In [234]:
from pyspark.sql import SparkSession
from pyspark.sql.types import DoubleType
from pyspark.sql import functions as F
from pyspark.sql.functions import lit, col
from dateutil.relativedelta import relativedelta
from pyspark.sql.types import StructType, StructField, StringType, LongType
from pyspark.sql.functions import expr, col, round, avg, max, min, sum, count, when, lit
from itertools import product
from datetime import datetime
import pytz
import os
import logging

In [235]:

# Configura logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)


In [236]:
def configure_spark():

    spark = (
        SparkSession.builder
        .appName("lakehouse-jupyter-book-avaliacoes")

        # conexão com cluster
        .master(os.getenv("SPARK_MASTER"))

        # 🚨 ESSENCIAL em Docker
        .config("spark.driver.host", os.getenv("SPARK_DRIVER_HOST"))
        .config("spark.driver.bindAddress", "0.0.0.0")

        # Delta Lake
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

        # S3 / MinIO
        .config("spark.hadoop.fs.s3a.endpoint", os.getenv("MINIO_ENDPOINT"))
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")

        .config("spark.hadoop.fs.s3a.access.key", os.getenv("AWS_ACCESS_KEY_ID"))
        .config("spark.hadoop.fs.s3a.secret.key", os.getenv("AWS_SECRET_ACCESS_KEY"))

        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config(
            "spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
        )

        # performance
        .config("spark.sql.adaptive.enabled", "true")

        # Delta fix
        .config("spark.databricks.delta.retentionDurationCheck.enabled", "false")

        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("WARN")

    return spark

In [237]:
agora=datetime.now(pytz.timezone('America/Sao_Paulo'))
dthproc=agora.strftime("%Y%m%d%H%M%S")

In [238]:
#executar todas as datas abaixo em (  data_exec_inicial  ) 
#201808, 201807, 201806, 201805, 201804, 201803, 201802, 201801, 201712, 201711, 201710, 201709
# converte YYYYMM -> date
data_exec_inicial = 201808
# converte YYYYMM -> date
data_dt = datetime.strptime(str(data_exec_inicial), "%Y%m")

# subtrai 12 meses
data_exec_final = int((data_dt - relativedelta(months=12)).strftime("%Y%m"))
data_exec_final

201708

In [239]:
spark = configure_spark()
spark

In [240]:
logging.info("Iniciando criação do book: pedidos")

df_orders = (
    spark.read.table("delta.`s3a://silver/olist_orders`")
)

df_orders.createOrReplaceTempView("df_orders")

2026-06-21 14:37:47,829 - INFO - Iniciando criação do book: pedidos


In [241]:
df_order_items = spark.read.table("delta.`s3a://silver/olist_order_items`")

df_order_items.createOrReplaceTempView("df_order_items")

In [242]:
olist_order_reviews = spark.read.table("delta.`s3a://silver/olist_order_reviews`")

olist_order_reviews.createOrReplaceTempView("olist_order_reviews")

In [243]:
df_orders_01 = spark.sql(f"""
    SELECT DISTINCT B.seller_id, A.safra 
    FROM df_orders AS A
    LEFT JOIN df_order_items AS B
    ON A.order_id = B.order_id
    WHERE A.safra = {data_exec_inicial}
""")

df_orders_01.createOrReplaceTempView("df_orders_01")

In [244]:
df_join = spark.sql(f"""
-- ============================================================
-- VARIÁVEIS DE REVIEWS PARA ANÁLISE DE CHURN DE SELLERS
-- ============================================================
-- PERÍODO: {data_exec_final} A {data_exec_inicial}
-- OBJETIVO: Criar variáveis derivadas da tabela REVIEWS
-- GRANULARIDADE: seller_id, safra
-- ============================================================

WITH base_orders_reviews AS (
  -- CTE 1: Junta todas as tabelas base, filtrando apenas pedidos entregues
  SELECT
    oi.seller_id,
    o.order_id,
    o.safra,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    r.review_id,
    r.review_score,
    r.review_comment_title,
    r.review_comment_message
  FROM df_orders o
  INNER JOIN df_order_items oi ON o.order_id = oi.order_id
  LEFT JOIN olist_order_reviews r ON o.order_id = r.order_id
  WHERE o.order_status = 'DELIVERED'
    AND o.order_delivered_customer_date IS NOT NULL
    AND o.order_estimated_delivery_date IS NOT NULL
    AND o.safra BETWEEN {data_exec_final} AND {data_exec_inicial}
),

seller_safra_vars AS (
  -- CTE 2: Agrega as variáveis por seller_id e safra
  SELECT
    seller_id,
    safra,
    
    -- Grupo 1: Volume e Engajamento
    COUNT(DISTINCT review_id) AS qtd_reviews,
    COUNT(DISTINCT CASE 
      WHEN (review_comment_title IS NOT NULL AND TRIM(review_comment_title) != '') 
        OR (review_comment_message IS NOT NULL AND TRIM(review_comment_message) != '') 
      THEN review_id 
    END) AS qtd_reviews_comentadas,
    SUM(CASE 
      WHEN (review_comment_title IS NOT NULL AND TRIM(review_comment_title) != '') 
        OR (review_comment_message IS NOT NULL AND TRIM(review_comment_message) != '') 
      THEN 1 ELSE 0 
    END) * 1.0 / NULLIF(COUNT(DISTINCT review_id), 0) AS pct_reviews_comentadas,
    COUNT(DISTINCT CASE 
      WHEN (review_comment_title IS NULL OR TRIM(review_comment_title) = '') 
        AND (review_comment_message IS NULL OR TRIM(review_comment_message) = '') 
      THEN review_id 
    END) AS qtd_reviews_sem_comentario,
    SUM(CASE 
      WHEN (review_comment_title IS NULL OR TRIM(review_comment_title) = '') 
        AND (review_comment_message IS NULL OR TRIM(review_comment_message) = '') 
      THEN 1 ELSE 0 
    END) * 1.0 / NULLIF(COUNT(DISTINCT review_id), 0) AS pct_reviews_sem_comentario,

    -- Grupo 2: Sentimento e Satisfação
    AVG(review_score) AS avg_review_score,
    PERCENTILE_APPROX(review_score, 0.5) AS median_review_score,
    STDDEV_POP(review_score) AS std_review_score,
    MIN(review_score) AS min_review_score,
    MAX(review_score) AS max_review_score,
    MAX(review_score) - MIN(review_score) AS amplitude_review,
    STDDEV_POP(review_score) / NULLIF(AVG(review_score), 0) AS coeficiente_variacao_review,

    -- Grupo 3: Notas Baixas e Negatividade
    COUNT(DISTINCT CASE WHEN review_score = 1 THEN review_id END) AS qtd_review_1,
    COUNT(DISTINCT CASE WHEN review_score = 2 THEN review_id END) AS qtd_review_2,
    COUNT(DISTINCT CASE WHEN review_score <= 2 THEN review_id END) AS qtd_review_negativa,
    SUM(CASE WHEN review_score = 1 THEN 1 ELSE 0 END) * 1.0 / NULLIF(COUNT(DISTINCT review_id), 0) AS pct_review_1,
    SUM(CASE WHEN review_score = 2 THEN 1 ELSE 0 END) * 1.0 / NULLIF(COUNT(DISTINCT review_id), 0) AS pct_review_2,
    SUM(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END) * 1.0 / NULLIF(COUNT(DISTINCT review_id), 0) AS pct_review_negativa,
    COUNT(DISTINCT CASE WHEN review_score = 3 THEN review_id END) AS qtd_review_3,
    SUM(CASE WHEN review_score = 3 THEN 1 ELSE 0 END) * 1.0 / NULLIF(COUNT(DISTINCT review_id), 0) AS pct_review_3,

    -- Grupo 4: Notas Altas e Promoção
    COUNT(DISTINCT CASE WHEN review_score = 5 THEN review_id END) AS qtd_review_5,
    COUNT(DISTINCT CASE WHEN review_score >= 4 THEN review_id END) AS qtd_review_4_5,
    SUM(CASE WHEN review_score = 5 THEN 1 ELSE 0 END) * 1.0 / NULLIF(COUNT(DISTINCT review_id), 0) AS pct_review_5,
    SUM(CASE WHEN review_score >= 4 THEN 1 ELSE 0 END) * 1.0 / NULLIF(COUNT(DISTINCT review_id), 0) AS pct_review_4_5,
    SUM(CASE WHEN review_score = 4 THEN 1 ELSE 0 END) * 1.0 / NULLIF(COUNT(DISTINCT review_id), 0) AS pct_review_4,

    -- Grupo 5: Análise NPS
    COUNT(DISTINCT CASE WHEN review_score = 5 THEN review_id END) AS qtd_promotores,
    COUNT(DISTINCT CASE WHEN review_score = 4 THEN review_id END) AS qtd_neutros,
    COUNT(DISTINCT CASE WHEN review_score <= 3 THEN review_id END) AS qtd_detratores,
    SUM(CASE WHEN review_score = 5 THEN 1 ELSE 0 END) * 1.0 / NULLIF(COUNT(DISTINCT review_id), 0) AS pct_promotores,
    SUM(CASE WHEN review_score = 4 THEN 1 ELSE 0 END) * 1.0 / NULLIF(COUNT(DISTINCT review_id), 0) AS pct_neutros,
    SUM(CASE WHEN review_score <= 3 THEN 1 ELSE 0 END) * 1.0 / NULLIF(COUNT(DISTINCT review_id), 0) AS pct_detratores,
    ( (SUM(CASE WHEN review_score = 5 THEN 1 ELSE 0 END) - SUM(CASE WHEN review_score <= 3 THEN 1 ELSE 0 END)) * 100.0 ) 
      / NULLIF(COUNT(DISTINCT review_id), 0) AS nps_proxy,

    -- Grupo 6: Experiência de Entrega e Reviews
    COUNT(DISTINCT CASE 
      WHEN review_score <= 2 AND order_delivered_customer_date > order_estimated_delivery_date 
      THEN review_id 
    END) AS qtd_reviews_negativas_com_atraso,
    SUM(CASE 
      WHEN review_score <= 2 AND order_delivered_customer_date > order_estimated_delivery_date 
      THEN 1 ELSE 0 
    END) * 1.0 / NULLIF(SUM(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END), 0) AS pct_reviews_negativas_com_atraso,
    AVG(CASE 
      WHEN review_score <= 2 
      THEN DATEDIFF(order_delivered_customer_date, order_estimated_delivery_date) 
    END) AS avg_atraso_entrega_reviews_negativas
    
  FROM base_orders_reviews
  GROUP BY seller_id, safra
)

-- SELECT Final com todas as variáveis e flags
SELECT
  seller_id,
  safra,
  
  -- Grupo 1: Volume e Engajamento
  qtd_reviews,
  qtd_reviews_comentadas,
  pct_reviews_comentadas,
  qtd_reviews_sem_comentario,
  pct_reviews_sem_comentario,
  
  -- Grupo 2: Sentimento e Satisfação
  avg_review_score,
  median_review_score,
  std_review_score,
  min_review_score,
  max_review_score,
  amplitude_review,
  coeficiente_variacao_review,
  
  -- Grupo 3: Notas Baixas e Negatividade
  qtd_review_1,
  qtd_review_2,
  qtd_review_negativa,
  pct_review_1,
  pct_review_2,
  pct_review_negativa,
  qtd_review_3,
  pct_review_3,
  
  -- Grupo 4: Notas Altas e Promoção
  qtd_review_5,
  qtd_review_4_5,
  pct_review_5,
  pct_review_4_5,
  pct_review_4,
  
  -- Grupo 5: Análise NPS
  qtd_promotores,
  qtd_neutros,
  qtd_detratores,
  pct_promotores,
  pct_neutros,
  pct_detratores,
  nps_proxy,
  
  -- Grupo 6: Experiência de Entrega
  qtd_reviews_negativas_com_atraso,
  pct_reviews_negativas_com_atraso,
  avg_atraso_entrega_reviews_negativas,

  
  -- Grupo 7: Flags Binárias
  CASE WHEN avg_review_score < 3.5 THEN 1 ELSE 0 END AS flag_review_media_baixa,
  CASE WHEN pct_detratores > 0.30 THEN 1 ELSE 0 END AS flag_muitos_detratores,
  CASE WHEN pct_review_negativa > 0.20 THEN 1 ELSE 0 END AS flag_muitos_reviews_negativos,
  CASE WHEN avg_review_score < 4.0 THEN 1 ELSE 0 END AS flag_baixa_reputacao,
  CASE WHEN amplitude_review >= 4 THEN 1 ELSE 0 END AS flag_alta_amplitude

FROM seller_safra_vars
ORDER BY seller_id, safra
""")

df_join.createOrReplaceTempView("df_temp_01")

In [245]:
df_join.createOrReplaceTempView("df_transacoes")

In [246]:
df_temp_01 = spark.sql("""
WITH base AS (
    SELECT
        *,
        TO_DATE(CONCAT(safra, '01'), 'yyyyMMdd') AS data_dt
    FROM df_transacoes
)

SELECT
    *,
    CASE
        WHEN data_dt BETWEEN
             ADD_MONTHS(MAX(data_dt) OVER (PARTITION BY seller_id), -1)
             AND MAX(data_dt) OVER (PARTITION BY seller_id)
        THEN 1 ELSE 0
    END AS u1m,

    CASE
        WHEN data_dt BETWEEN
             ADD_MONTHS(MAX(data_dt) OVER (PARTITION BY seller_id), -3)
             AND MAX(data_dt) OVER (PARTITION BY seller_id)
        THEN 1 ELSE 0
    END AS u3m,


    CASE
        WHEN data_dt BETWEEN
             ADD_MONTHS(MAX(data_dt) OVER (PARTITION BY seller_id), -6)
             AND MAX(data_dt) OVER (PARTITION BY seller_id)
        THEN 1 ELSE 0
    END AS u6m,

    CASE
        WHEN data_dt BETWEEN
             ADD_MONTHS(MAX(data_dt) OVER (PARTITION BY seller_id), -9)
             AND MAX(data_dt) OVER (PARTITION BY seller_id)
        THEN 1 ELSE 0
    END AS u9m,    

    CASE
        WHEN data_dt BETWEEN
             ADD_MONTHS(MAX(data_dt) OVER (PARTITION BY seller_id), -12)
             AND MAX(data_dt) OVER (PARTITION BY seller_id)
        THEN 1 ELSE 0
    END AS u12m

FROM base
ORDER BY seller_id, safra
""")

df_temp_01.createOrReplaceTempView("df_temp_01")

In [247]:
# Definição das variáveis
coluna_chave = "seller_id"
colunas_flags = ['u1m', 'u3m', 'u6m', 'u9m', 'u12m']

# Lista de colunas de valores
colunas_valores = [
    'qtd_reviews', 'qtd_reviews_comentadas',  'pct_reviews_comentadas', 'qtd_reviews_sem_comentario',  'pct_reviews_sem_comentario', 'avg_review_score', 'median_review_score', 'std_review_score',  'min_review_score',
    'max_review_score', 'amplitude_review', 'coeficiente_variacao_review', 'qtd_review_1',  'qtd_review_2', 'qtd_review_negativa',  'pct_review_1', 'pct_review_2', 'pct_review_negativa', 'qtd_review_3', 'pct_review_3',
    'qtd_review_5',  'qtd_review_4_5', 'pct_review_5',  'pct_review_4_5',  'pct_review_4',  'qtd_promotores',  'qtd_neutros',  'qtd_detratores',  'pct_promotores',  'pct_neutros',  'pct_detratores',  'nps_proxy',  'qtd_reviews_negativas_com_atraso',
    'pct_reviews_negativas_com_atraso',  'avg_atraso_entrega_reviews_negativas',  'flag_review_media_baixa', 'flag_muitos_detratores',  'flag_muitos_reviews_negativos',  'flag_baixa_reputacao',   'flag_alta_amplitude'
]

# Configuração dos indicadores
"""indicadores_config = {
    'IND_PDD': {'alias': 'PDD', 'valores': ['S']},
    'IND_WO': {'alias': 'WO', 'valores': ['R']},
    'IND_PCCR': {'alias': 'PCCR', 'valores': ['W']},
    'IND_ACA': {'alias': 'ACA', 'valores': ['N']},
    'IND_PRIMEIRA_FAT': {'alias': 'PRIM_FAT', 'valores': ['S']},
    'IND_FRAUDE': {'alias': 'FRAUDE', 'valores': ['N']}
}"""



def gerar_sql_dinamico():
    #selects = ["seller_id", "seller_region"]
    selects = ["seller_id"]
    
    # Agregações básicas
    for flag in colunas_flags:
        for valor in colunas_valores:
            selects.append(f"CAST(round(avg(case when {flag} = 1 then {valor} else NULL end), 2) AS DOUBLE) as vl_med_{flag}_{valor}_reviews")
            selects.append(f"CAST(round(max(case when {flag} = 1 then {valor} else NULL end), 2) AS DOUBLE) as vl_max_{flag}_{valor}_reviews")
            selects.append(f"CAST(round(min(case when {flag} = 1 then {valor} else NULL end), 2) AS DOUBLE) as vl_min_{flag}_{valor}_reviews")
            #selects.append(f"CAST(round(stddev(case when {flag} = 1 then {valor} else NULL end), 2) AS DOUBLE) as vl_std_{flag}_{valor}_reviews")
            #selects.append(f"CAST(round(count(case when {flag} = 1 then {valor} else NULL end), 2) AS DOUBLE) as vl_qtd_{flag}_{valor}_reviews")
    # Agregações com indicadores
    """for indicador, info in indicadores_config.items():
        for valor_indicador in info['valores']:
            for flag in colunas_flags:
                for valor in colunas_valores:
                    alias = info['alias']
                    selects.append(f"round(avg(case when {flag} = 1 and {indicador} = '{valor_indicador}' then {valor} else NULL end), 2) as vl_med_{flag}_{alias}_{valor_indicador}_{valor}_orders")
                    #selects.append(f"round(max(case when {flag} = 1 and {indicador} = '{valor_indicador}' then {valor} else NULL end), 2) as vl_max_{flag}_{alias}_{valor_indicador}_{valor}_orders")
                    #selects.append(f"round(count(case when {flag} = 1 and {indicador} = '{valor_indicador}' then {valor} else NULL end), 2) as vl_qtd_{flag}_{alias}_{valor_indicador}_{valor}_orders")"""
    
    sql_query = f"""
    SELECT
        {', '.join(selects)}
    FROM df_temp_01
    GROUP BY seller_id
    ORDER BY seller_id
    """
    
    return sql_query

# Executar SQL dinâmico
sql_dinamico = gerar_sql_dinamico()

df_temp_02 = spark.sql(sql_dinamico)

df_temp_02.createOrReplaceTempView("df_temp_02")
print(f"Shape: {df_temp_02.count()} linhas, {len(df_temp_02.columns)} colunas")

Shape: 2657 linhas, 601 colunas


In [248]:
def add_temporal_ratio_columns(
    df,
    base_prefix="vl_med",
    ratio_prefix="raz_med",
    windows=("u1m", "u3m", "u6m", "u9m", "u12m"),
    suffix="_reviews"
):
    """
    Função que mantém todas as colunas originais do DataFrame
    e adiciona novas colunas de razão temporal entre janelas consecutivas.

    Exemplo de razão criada:
    RAZ_MED_U1M_U3M_FAT_ATRASO = VL_MED_U1M_FAT_ATRASO / VL_MED_U3M_FAT_ATRASO
    """

    # Cria uma lista com todas as colunas originais do DataFrame
    # Isso garante que nenhuma coluna existente será removida
    base_cols = [F.col(c) for c in df.columns]

    # Lista que armazenará as expressões das novas colunas de razão
    ratio_exprs = []

    # Conjunto com os nomes das colunas do DataFrame
    # Usado para checar rapidamente se uma coluna existe
    df_cols = set(df.columns)

    # Gera pares de janelas consecutivas
    # Exemplo: (U1M, U3M), (U3M, U6M), ...
    window_pairs = list(zip(windows[:-1], windows[1:]))

    # Percorre cada par de janelas (numerador e denominador)
    for num_win, den_win in window_pairs:

        # Percorre todas as colunas do DataFrame
        for col_num in df.columns:

            # Verifica se a coluna pertence à janela do numerador
            # e segue o padrão: VL_MED__
            if not col_num.startswith(f"{base_prefix}_{num_win}_"):
                continue

            # Deriva o nome da coluna do denominador
            # Substitui a janela do numerador pela janela do denominador
            col_den = col_num.replace(
                f"{base_prefix}_{num_win}_",
                f"{base_prefix}_{den_win}_"
            )

            # Se a coluna do denominador não existir, ignora
            if col_den not in df_cols:
                continue

            # Extrai o nome da feature base
            # Remove prefixo (VL_MED__) e o sufixo (_ATRASO)
            feature = (
                col_num
                .replace(f"{base_prefix}_{num_win}_", "")
                .replace(suffix, "")
            )

            # Define o nome da nova coluna de razão temporal
            # Exemplo: RAZ_MED_U1M_U3M_FAT_ATRASO
            ratio_name = (
                f"{ratio_prefix}_{num_win}_{den_win}_"
                f"{feature}{suffix}"
            )

            # Cria a expressão da razão temporal
            # Realiza a divisão apenas quando o denominador é diferente de zero
            # Caso contrário, retorna NULL
            ratio_exprs.append(
                F.when(
                    F.col(col_den) != 0,
                    F.col(col_num) / F.col(col_den)
                )
                .cast(DoubleType())  # Garantir que seja DOUBLE
                .alias(ratio_name)
            )


    # Retorna o DataFrame mantendo todas as colunas originais
    # e adicionando as novas colunas de razão temporal
    return df.select(*base_cols, *ratio_exprs)


# Aplica a função ao DataFrame anterior, gerando um novo DataFrame enriquecido
df_temp_03 = add_temporal_ratio_columns(df_temp_02)


df_temp_03.createOrReplaceTempView("df_temp_03")

print(f"Shape: {df_temp_03.count()} linhas, {len(df_temp_03.columns)} colunas")

Shape: 2657 linhas, 761 colunas
root
 |-- seller_id: string (nullable = true)
 |-- vl_med_u1m_qtd_reviews_reviews: double (nullable = true)
 |-- vl_max_u1m_qtd_reviews_reviews: double (nullable = true)
 |-- vl_min_u1m_qtd_reviews_reviews: double (nullable = true)
 |-- vl_med_u1m_qtd_reviews_comentadas_reviews: double (nullable = true)
 |-- vl_max_u1m_qtd_reviews_comentadas_reviews: double (nullable = true)
 |-- vl_min_u1m_qtd_reviews_comentadas_reviews: double (nullable = true)
 |-- vl_med_u1m_pct_reviews_comentadas_reviews: double (nullable = true)
 |-- vl_max_u1m_pct_reviews_comentadas_reviews: double (nullable = true)
 |-- vl_min_u1m_pct_reviews_comentadas_reviews: double (nullable = true)
 |-- vl_med_u1m_qtd_reviews_sem_comentario_reviews: double (nullable = true)
 |-- vl_max_u1m_qtd_reviews_sem_comentario_reviews: double (nullable = true)
 |-- vl_min_u1m_qtd_reviews_sem_comentario_reviews: double (nullable = true)
 |-- vl_med_u1m_pct_reviews_sem_comentario_reviews: double (nullabl

In [249]:
df_temp_04 = df_orders_01.alias("t1") \
    .join(df_temp_03.alias("t2"), "seller_id", "left") \
    .withColumn("safra", lit(data_exec_inicial)) \
    .withColumn("datproc", lit(dthproc)) \

df_temp_04.createOrReplaceTempView("df_temp_04")
df_temp_04.count()

1264

In [250]:
spark.sql("""
-- 4) Calcula hash para idempotência (evita UPDATE sem mudança real)
CREATE OR REPLACE TEMP VIEW stage_orders_final AS
SELECT
  *,
  sha2(concat_ws('||',
    coalesce(seller_id,''),
    coalesce(cast(safra as int),''),
    coalesce(cast(vl_med_u1m_qtd_reviews_reviews as int),'')
    --coalesce(product_id,''),
    --coalesce(seller_id,''),
    --coalesce(date_format(shipping_limit_date,'yyyy-MM-dd'),''),
    --coalesce(cast(price as string),''),
    --coalesce(cast(freight_value as string),'')
  ), 256) AS row_hash
FROM df_temp_04;
""")


DataFrame[]

In [251]:
spark.sql("""
        CREATE DATABASE IF NOT EXISTS gold
        LOCATION 's3a://gold'
""")

DataFrame[]

In [252]:
from delta.tables import DeltaTable

path = "s3a://gold/book_reviews"

if not spark.catalog.tableExists("gold.book_reviews"):
    
    # Se já existe Delta no storage → só registra
    if DeltaTable.isDeltaTable(spark, path):
        spark.sql(f"""
            CREATE TABLE gold.book_reviews
            USING DELTA
            LOCATION '{path}'
        """)
    
    # Se não existe nada → cria do zero
    else:
        spark.sql(f"""
            CREATE TABLE gold.book_reviews
            USING DELTA
            PARTITIONED BY (safra)
            LOCATION '{path}'
            AS SELECT * FROM stage_orders_final WHERE 1=0
        """)


In [253]:
spark.sql("""
MERGE INTO gold.book_reviews AS t
USING stage_orders_final AS s
ON t.seller_id = s.seller_id
AND t.safra = s.safra

WHEN MATCHED AND (t.row_hash IS NULL OR t.row_hash <> s.row_hash) THEN
UPDATE SET *

WHEN NOT MATCHED THEN
INSERT *
;
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [254]:
logging.info("Finalizando criação do book: reviews")
spark.stop()

2026-06-21 14:38:53,293 - INFO - Finalizando criação do book: reviews
